# FloodShift 07 — Deterministic GeoEngine + Grounded Evidence Packet

This stage converts the accepted FloodShift detector outputs into **machine-verifiable numerical facts** for the later grounded VLM.

It does **not train** the segmentation model and does **not generate natural-language explanations**.

The accepted detector remains:

\[
\boxed{\text{04H-CAL}}
\]

with:

- exact 04H checkpoint lineage,
- fixed water-type bias \(b=-0.15\),
- frozen Notebook-05 temperature and entropy review thresholds,
- frozen Notebook-06 counterfactual attribution thresholds.

## Core design rule

The evidence packet is deployment-style.

It may contain:

- sensor-derived facts,
- segmentation-derived facts,
- calibrated uncertainty,
- counterfactual SAR/optical dependence,
- geospatial extent,
- deterministic connected-component geometry,
- provenance and verifier hashes.

It must **not** contain:

- ground-truth labels,
- test-set information,
- an LLM-generated explanation,
- causal claims about why flooding occurred,
- claims that counterfactual attribution proves physical causality.

Ground truth is used only in a separate **integrity audit** to prove that the frozen inference path still reproduces Notebook 06. It is never written into the VLM-facing packet.

## Packet families

For each attribution scene the GeoEngine produces:

1. **Coverage**
   - sensor-valid coverage fraction
   - geospatial valid area when raster CRS supports metric/geodesic area

2. **Segmentation**
   - non-flood fraction/area
   - permanent-water fraction/area
   - temporary-flood fraction/area
   - total predicted-water fraction/area

3. **Uncertainty**
   - mean / median / p90 normalized predictive entropy
   - frozen review-10 and review-20 pixel fractions
   - temporary-flood uncertainty

4. **Sensor quality**
   - mean \(P(\mathrm{clear})\)
   - fraction with \(P(\mathrm{clear})<0.5\)

5. **Counterfactual attribution**
   - normalized JS sensitivity to removing SAR
   - normalized JS sensitivity to removing optical
   - SAR/optical contribution shares
   - prediction flip rates
   - temporary-flood-specific sensor attribution
   - high-fragility tile fraction using the frozen Notebook-06 threshold

6. **Spatial geometry**
   - number of predicted temporary-flood connected components
   - largest component size/area
   - largest component centroid in WGS84 where georeferencing is available

7. **Provenance**
   - accepted checkpoint SHA256
   - frozen calibration constants
   - packet SHA256
   - fact schema version

The output is intended to be consumed by the next stage:

\[
\boxed{\text{Grounded instruction dataset}}
\]

and later by the variational VLM.


In [ ]:
# ============================================================
# CELL: P15-C00 — KAGGLE-COMPAT TERRATORCH INSTALL
# ============================================================
# Proven compatible path from 04E.
# IMPORTANT: on a fresh Kaggle image, pip may change NumPy to 1.26.4.
# If the next cell reports a NumPy/Pandas ABI error, restart the Python
# kernel ONCE, do NOT rerun this cell, then continue from P10-C01.

import sys, subprocess

print("Installing TerraTorch 1.1.1 (04E/04F-proven compatibility path)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-q", "--no-cache-dir", "terratorch==1.1.1"
])
print("P15-C00 COMPLETE ✅")
print("If NumPy changed, restart the Python kernel once before P10-C01.")

In [1]:
# ============================================================
# CELL: P15-C01 — ENVIRONMENT + GEOENGINE DEPENDENCIES
# ============================================================
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import sys, gc, json, time, math, random, hashlib, zipfile, warnings
from pathlib import Path

import numpy as np
import scipy
from scipy import ndimage
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import rasterio
from rasterio.windows import Window
from rasterio.warp import transform as rio_transform

try:
    from pyproj import CRS
    PYPROJ_AVAILABLE = True
except Exception as e:
    PYPROJ_AVAILABLE = False
    warnings.warn(f"pyproj unavailable; metric area may be omitted: {e}")

import matplotlib.pyplot as plt

import terratorch
from terratorch.registry import BACKBONE_REGISTRY

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = torch.cuda.is_available()
torch.backends.cudnn.benchmark = True

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/FloodShift_07_GeoEngine")
PACKET_ROOT = OUTPUT_ROOT / "packets"
PANEL_ROOT = OUTPUT_ROOT / "panels"
METRIC_ROOT = OUTPUT_ROOT / "metrics"

for p in [OUTPUT_ROOT, PACKET_ROOT, PANEL_ROOT, METRIC_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("=" * 112)
print("P15-C01 — GEOENGINE ENVIRONMENT")
print("=" * 112)
print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("SciPy       :", scipy.__version__)
print("PyTorch     :", torch.__version__)
print("Rasterio    :", rasterio.__version__)
print("pyproj      :", PYPROJ_AVAILABLE)
print("Device      :", DEVICE)
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
print("Output root :", OUTPUT_ROOT)


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


P15-C01 — GEOENGINE ENVIRONMENT
Python      : 3.12.13
NumPy       : 1.26.4
Pandas      : 2.3.3
SciPy       : 1.16.3
PyTorch     : 2.10.0+cu128
Rasterio    : 1.4.4
pyproj      : True
Device      : cuda:0
GPU         : Tesla T4
Output root : /kaggle/working/FloodShift_07_GeoEngine


In [4]:
# ============================================================
# CELL: P15-C02 — LOCATE + VERIFY 04H-CAL / 05 / 06 PROVENANCE
# ============================================================
_file_cache = {}

def locate_by_saved_path(saved_path_or_name, required=True):
    filename = Path(str(saved_path_or_name)).name
    key = str(saved_path_or_name)

    if key in _file_cache:
        return _file_cache[key]

    matches = []
    if INPUT_ROOT.exists():
        matches.extend(INPUT_ROOT.rglob(filename))

    if not matches:
        if required:
            raise FileNotFoundError(
                f"Could not locate '{filename}' under /kaggle/input."
            )
        return None

    selected = sorted(matches, key=lambda p: (len(str(p)), str(p)))[0]
    _file_cache[key] = selected
    return selected

def resolve_from_archive(filename, archive_name, required=True):
    direct = locate_by_saved_path(filename, required=False)
    if direct is not None:
        return direct

    archive = locate_by_saved_path(archive_name, required=False)
    if archive is None:
        if required:
            raise FileNotFoundError(
                f"Need either '{filename}' or '{archive_name}' in /kaggle/input."
            )
        return None

    out_dir = OUTPUT_ROOT / "_extracted_inputs" / Path(archive_name).stem
    out_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(archive, "r") as zf:
        zf.extractall(out_dir)

    matches = list(out_dir.rglob(filename))
    if not matches:
        if required:
            raise FileNotFoundError(
                f"'{filename}' not found inside '{archive_name}'."
            )
        return None

    return matches[0]

FINAL_MANIFEST_PATH = locate_by_saved_path("floodshift_preprocessing_manifest_v1.csv")
GLOBAL_LOCK_PATH = locate_by_saved_path("floodshift_preprocessing_GLOBAL_LOCK_v1.json")
OPTICAL_NORM_PATH = locate_by_saved_path("optical_10band_train_normalization_exact.csv")
SAR_SCENE_STATS_PATH = locate_by_saved_path("train_raw_sar_scene_statistics.csv")

CHECKPOINT_04H = locate_by_saved_path("FloodShift_04H_best_mIoU.pt")
CAL_CONFIG_PATH = locate_by_saved_path("FloodShift_04H_CAL_config.pt")

UNCERTAINTY_05_PATH = resolve_from_archive(
    "FloodShift_05_uncertainty_config.pt",
    "FloodShift_05_ARCHIVE.zip",
    required=True,
)

ATTR_CONFIG_06_PATH = resolve_from_archive(
    "FloodShift_06_attribution_config.pt",
    "FloodShift_06_ARCHIVE.zip",
    required=True,
)

PROTOCOL_06_PATH = resolve_from_archive(
    "FloodShift_06_protocol_record.json",
    "FloodShift_06_ARCHIVE.zip",
    required=True,
)

EXPECTED_04H_SHA256 = "ef3341a7579f1316c80290dcd2a8f972c0adbf7e03e7a8c6bc239f30c4b47e53"
EXPECTED_04E_SHA256 = "6ab8ad8b41aa64b4ceee403539d8554a14f61cab235a0f0841ef13f77e390822"
EXPECTED_TYPE_BIAS = -0.15

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

SHA_04H = sha256_file(CHECKPOINT_04H)
assert SHA_04H == EXPECTED_04H_SHA256

cal_cfg = torch.load(
    CAL_CONFIG_PATH,
    map_location="cpu",
    weights_only=False,
)

unc05 = torch.load(
    UNCERTAINTY_05_PATH,
    map_location="cpu",
    weights_only=False,
)

attr06 = torch.load(
    ATTR_CONFIG_06_PATH,
    map_location="cpu",
    weights_only=False,
)

with open(PROTOCOL_06_PATH, "r") as f:
    protocol06 = json.load(f)

assert cal_cfg["source_04h_checkpoint_sha256"] == EXPECTED_04H_SHA256
assert abs(float(cal_cfg["type_bias"]) - EXPECTED_TYPE_BIAS) < 1e-12
assert cal_cfg["decision"] == "04H_CAL_SINGLE_MODEL_PREFERRED"

assert unc05["source_04h_sha256"] == EXPECTED_04H_SHA256
assert abs(float(unc05["type_bias"]) - EXPECTED_TYPE_BIAS) < 1e-12

assert attr06["source_04h_sha256"] == EXPECTED_04H_SHA256
assert abs(float(attr06["type_bias"]) - EXPECTED_TYPE_BIAS) < 1e-12
assert abs(float(attr06["temperature"]) - float(unc05["temperature"])) < 1e-9

TYPE_BIAS = float(cal_cfg["type_bias"])
TEMPERATURE = float(unc05["temperature"])
ENTROPY_THRESHOLD_REVIEW_10 = float(unc05["entropy_threshold_review_10"])
ENTROPY_THRESHOLD_REVIEW_20 = float(unc05["entropy_threshold_review_20"])
HIGH_FRAGILITY_THRESHOLD = float(attr06["high_fragility_threshold"])
DOMINANCE_SHARE = float(attr06["dominance_share_threshold"])

with open(GLOBAL_LOCK_PATH, "r") as f:
    preprocessing_lock = json.load(f)

assert preprocessing_lock["status"] == "PERMANENTLY_FROZEN"
assert preprocessing_lock["preprocessing_development_closed"] is True

print("=" * 112)
print("P15-C02 — FROZEN PROVENANCE VERIFIED")
print("=" * 112)
print("04H SHA256          :", SHA_04H)
print("Type bias           :", TYPE_BIAS)
print("Temperature         :", TEMPERATURE)
print("Review-10 entropy   :", ENTROPY_THRESHOLD_REVIEW_10)
print("Review-20 entropy   :", ENTROPY_THRESHOLD_REVIEW_20)
print("Fragility threshold :", HIGH_FRAGILITY_THRESHOLD)
print("Dominance threshold :", DOMINANCE_SHARE)
print("05B weights         : NOT USED")
print("Test status         : LOCKED_NOT_EVALUATED")
print("P15-C02 COMPLETE ✅")


P15-C02 — FROZEN PROVENANCE VERIFIED
04H SHA256          : ef3341a7579f1316c80290dcd2a8f972c0adbf7e03e7a8c6bc239f30c4b47e53
Type bias           : -0.15
Temperature         : 0.989592432975769
Review-10 entropy   : 0.1218614846467972
Review-20 entropy   : 0.04920305460691452
Fragility threshold : 0.20378518104553223
Dominance threshold : 0.65
05B weights         : NOT USED
Test status         : LOCKED_NOT_EVALUATED
P15-C02 COMPLETE ✅


In [5]:
# ============================================================
# CELL: P15-C03 — MANIFEST + FROZEN TRAIN-ONLY NORMALIZATION
# ============================================================

manifest = pd.read_csv(FINAL_MANIFEST_PATH)

assert len(manifest) == 440
assert manifest["scene_id"].nunique() == 440

train_df = (
    manifest[manifest["split"] == "train"]
    .copy()
    .reset_index(drop=True)
)

val_df = (
    manifest[manifest["split"] == "validation"]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    manifest[manifest["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)

assert len(train_df) == 314
assert len(val_df) == 68
assert len(test_df) == 58

optical_norm = pd.read_csv(OPTICAL_NORM_PATH)

OPTICAL_BANDS = optical_norm["band_name"].tolist()

EXPECTED_OPTICAL_BANDS = [
    "B2", "B3", "B4", "B5", "B6",
    "B7", "B8", "B8A", "B11", "B12"
]

assert OPTICAL_BANDS == EXPECTED_OPTICAL_BANDS

OPTICAL_MEAN = optical_norm["mean"].to_numpy(np.float32)
OPTICAL_STD = optical_norm["std"].to_numpy(np.float32)

# Original S2 source order:
# B1 B2 B3 B4 B5 B6 B7 B8 B8A B9 B10 B11 B12
OPTICAL_SOURCE_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

sar_stats = pd.read_csv(SAR_SCENE_STATS_PATH)
assert len(sar_stats) == 314

n_all = sar_stats["valid_pixels"].to_numpy(np.float64)

def pooled_mean_std(mean_col, std_col):
    mu_all = sar_stats[mean_col].to_numpy(np.float64)
    sd_all = sar_stats[std_col].to_numpy(np.float64)

    valid = (
        np.isfinite(mu_all)
        & np.isfinite(sd_all)
        & (n_all > 0)
    )

    n = n_all[valid]
    mu = mu_all[valid]
    sd = sd_all[valid]

    total_n = n.sum()
    mean = np.sum(n * mu) / total_n
    second = np.sum(n * (sd**2 + mu**2)) / total_n
    var = max(second - mean**2, 0.0)

    return float(mean), float(np.sqrt(var))

VV_MEAN, VV_STD = pooled_mean_std(
    "vv_mean_db",
    "vv_std_db",
)

VH_MEAN, VH_STD = pooled_mean_std(
    "vh_mean_db",
    "vh_std_db",
)

SAR_MEAN = np.array(
    [VV_MEAN, VH_MEAN],
    dtype=np.float32,
)

SAR_STD = np.array(
    [VV_STD, VH_STD],
    dtype=np.float32,
)

assert np.isfinite(SAR_MEAN).all()
assert np.isfinite(SAR_STD).all()
assert (SAR_STD > 0).all()

print("=" * 112)
print("P15-C03 — DATA CONFIG")
print("=" * 112)
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print("Optical bands      :", OPTICAL_BANDS)
print("SAR mean [VV,VH]   :", SAR_MEAN)
print("SAR std  [VV,VH]   :", SAR_STD)
print("Reliability        : frozen continuous P(clear)")
print("Primary SAR input  : raw VV/VH")

P15-C03 — DATA CONFIG
Train / Val / Test: 314 68 58
Optical bands      : ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
SAR mean [VV,VH]   : [-10.358468 -17.10712 ]
SAR std  [VV,VH]   : [4.161632  4.8134336]
Reliability        : frozen continuous P(clear)
Primary SAR input  : raw VV/VH


In [7]:
# ============================================================
# CELL: P15-C04 — TARGET + DATASET
#
# IMPORTANT:
# This intentionally preserves the CURRENT project's target mapping.
# It does NOT silently change label semantics mid-experiment.
# ============================================================

IGNORE_INDEX = 255
NUM_CLASSES = 3

CLASS_NAMES = [
    "non_flood",
    "permanent_water",
    "temporary_flood",
]

TILE_SIZE = 256
TILES_PER_SCENE = 4

TILE_OFFSETS = [
    (0, 0),
    (0, 256),
    (256, 0),
    (256, 256),
]


def resolve_scene_files(row):
    return {
        "sar": locate_by_saved_path(row["sar_raw_path"]),
        "optical": locate_by_saved_path(row["optical_raw_path"]),
        "flood": locate_by_saved_path(row["flood_label_path"]),
        "jrc": locate_by_saved_path(row["jrc_water_path"]),
        "reliability": locate_by_saved_path(
            row["optical_reliability_path"]
        ),
    }


def build_three_class_target(flood, jrc):
    flood = np.asarray(flood)
    jrc = np.asarray(jrc)

    target = np.full(
        flood.shape,
        IGNORE_INDEX,
        dtype=np.uint8,
    )

    valid = flood != -1

    target[valid] = 0

    # Current frozen project semantics:
    # any valid JRC permanent-water pixel -> class 1
    target[
        valid & (jrc == 1)
    ] = 1

    # flood-positive but not JRC permanent water -> class 2
    target[
        valid
        & (jrc != 1)
        & (flood == 1)
    ] = 2

    return target


def normalize_sar(sar):
    sar = sar.astype(
        np.float32,
        copy=False,
    )

    valid = np.isfinite(sar).all(axis=0)

    out = np.zeros_like(
        sar,
        dtype=np.float32,
    )

    for c in range(2):
        m = valid & np.isfinite(sar[c])

        out[c, m] = (
            sar[c, m] - SAR_MEAN[c]
        ) / SAR_STD[c]

    return out


def normalize_optical(optical_13):
    optical_13 = optical_13.astype(
        np.float32,
        copy=False,
    )

    valid = np.all(
        np.isfinite(optical_13)
        & (optical_13 != 0),
        axis=0,
    )

    reflectance = optical_13 / 10000.0

    optical_10 = reflectance[
        OPTICAL_SOURCE_INDICES
    ]

    out = (
        optical_10
        - OPTICAL_MEAN[:, None, None]
    ) / OPTICAL_STD[:, None, None]

    out = out.astype(
        np.float32,
        copy=False,
    )

    # Frozen Notebook-03 convention.
    out[:, ~valid] = 0.0

    return out


def augment_all(
    sar,
    optical,
    reliability,
    target,
):
    # D4-aligned augmentation only.
    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=2)
        optical = np.flip(optical, axis=2)
        reliability = np.flip(
            reliability,
            axis=1,
        )
        target = np.flip(
            target,
            axis=1,
        )

    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=1)
        optical = np.flip(optical, axis=1)
        reliability = np.flip(
            reliability,
            axis=0,
        )
        target = np.flip(
            target,
            axis=0,
        )

    k = np.random.randint(0, 4)

    if k:
        sar = np.rot90(
            sar,
            k=k,
            axes=(1, 2),
        )

        optical = np.rot90(
            optical,
            k=k,
            axes=(1, 2),
        )

        reliability = np.rot90(
            reliability,
            k=k,
            axes=(0, 1),
        )

        target = np.rot90(
            target,
            k=k,
            axes=(0, 1),
        )

    return (
        sar.copy(),
        optical.copy(),
        reliability.copy(),
        target.copy(),
    )


class FixedEvalDataset(Dataset):
    def __init__(
        self,
        dataframe,
        train_mode=False,
    ):
        self.df = dataframe.reset_index(
            drop=True
        )

        self.train_mode = train_mode

    def __len__(self):
        return (
            len(self.df)
            * TILES_PER_SCENE
        )

    def __getitem__(self, index):
        scene_index = (
            index // TILES_PER_SCENE
        )

        tile_index = (
            index % TILES_PER_SCENE
        )

        row = self.df.iloc[
            scene_index
        ]

        scene_id = str(
            row["scene_id"]
        )

        y0, x0 = TILE_OFFSETS[
            tile_index
        ]

        window = Window(
            x0,
            y0,
            TILE_SIZE,
            TILE_SIZE,
        )

        paths = resolve_scene_files(
            row
        )

        with rasterio.open(
            paths["sar"]
        ) as src:
            sar = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["optical"]
        ) as src:
            optical_13 = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["flood"]
        ) as src:
            flood = src.read(
                1,
                window=window,
            )

        with rasterio.open(
            paths["jrc"]
        ) as src:
            jrc = src.read(
                1,
                window=window,
            )

        reliability_full = np.load(
            paths["reliability"],
            mmap_mode="r",
        )

        reliability = np.asarray(
            reliability_full[
                y0:y0 + TILE_SIZE,
                x0:x0 + TILE_SIZE,
            ],
            dtype=np.float32,
        )

        sar = normalize_sar(sar)

        optical = normalize_optical(
            optical_13
        )

        target = build_three_class_target(
            flood,
            jrc,
        )

        reliability = np.clip(
            reliability,
            0.0,
            1.0,
        )

        if self.train_mode:
            (
                sar,
                optical,
                reliability,
                target,
            ) = augment_all(
                sar,
                optical,
                reliability,
                target,
            )

        return {
            "sar": torch.from_numpy(
                sar
            ),
            "optical": torch.from_numpy(
                optical
            ),
            "reliability": torch.from_numpy(
                reliability[
                    None,
                    ...,
                ]
            ),
            "target": torch.from_numpy(
                target.astype(
                    np.int64
                )
            ),
            "scene_id": scene_id,
            "tile_index": tile_index,
        }

In [8]:
# ============================================================
# CELL: P15-C05 — SAME NOTEBOOK-06 ATTRIBUTION HALF
#
# The target is retained ONLY for an isolated integrity audit.
# It is NEVER written into the grounded evidence packet.
# ============================================================
VAL_BATCH_SIZE = 2
NUM_WORKERS = 2

def stable_scene_hash(scene_id):
    token = ("FloodShift05|" + str(scene_id)).encode("utf-8")
    return hashlib.sha256(token).hexdigest()

val_partition = val_df[["scene_id"]].copy()
val_partition["partition_hash"] = val_partition["scene_id"].map(stable_scene_hash)
val_partition = val_partition.sort_values("partition_hash").reset_index(drop=True)

calibration_ids = set(val_partition.iloc[:34]["scene_id"].astype(str))
attribution_ids = set(val_partition.iloc[34:]["scene_id"].astype(str))

val_attr_df = (
    val_df[
        val_df["scene_id"].astype(str).isin(attribution_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(val_attr_df) == 34
assert len(test_df) == 58

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

val_attr_loader = DataLoader(
    FixedEvalDataset(val_attr_df, train_mode=False),
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker,
    persistent_workers=(NUM_WORKERS > 0),
)

scene_row_lookup = {
    str(r["scene_id"]): r
    for _, r in val_attr_df.iterrows()
}

print("=" * 112)
print("P15-C05 — GEOENGINE INPUT SPLIT")
print("=" * 112)
print("Attribution scenes :", len(val_attr_df))
print("Tiles              :", len(val_attr_df) * TILES_PER_SCENE)
print("Frozen test scenes :", len(test_df), "(NO Dataset/DataLoader constructed)")
print("P15-C05 COMPLETE ✅")


P15-C05 — GEOENGINE INPUT SPLIT
Attribution scenes : 34
Tiles              : 136
Frozen test scenes : 58 (NO Dataset/DataLoader constructed)
P15-C05 COMPLETE ✅


In [12]:
# ============================================================
# CELL: P15-C06 — FINAL TERRAMIND-RIFT ARCHITECTURE
# ============================================================

# ------------------------------------------------------------
# TerraMind pretrained input conventions
# Official Sen1Floods11 config statistics.
# ------------------------------------------------------------

TM_S2_ALL_MEAN = np.array(
    [
        2357.089,
        2137.385,
        2018.788,
        2082.986,
        2295.651,
        2854.537,
        3122.849,
        3040.560,
        3306.481,
        1473.847,
        506.070,
        2472.825,
        1838.929,
    ],
    dtype=np.float32,
)

TM_S2_ALL_STD = np.array(
    [
        1624.683,
        1675.806,
        1557.708,
        1833.702,
        1823.738,
        1733.977,
        1732.131,
        1679.732,
        1727.260,
        1024.687,
        442.165,
        1331.411,
        1160.419,
    ],
    dtype=np.float32,
)

TM_S2_SUBSET_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

TM_S2_MEAN = (
    TM_S2_ALL_MEAN[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S2_STD = (
    TM_S2_ALL_STD[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S1_MEAN = np.array(
    [-12.599, -20.293],
    dtype=np.float32,
)

TM_S1_STD = np.array(
    [5.195, 5.890],
    dtype=np.float32,
)

TM_OPTICAL_BAND_NAMES = [
    "BLUE",
    "GREEN",
    "RED",
    "RED_EDGE_1",
    "RED_EDGE_2",
    "RED_EDGE_3",
    "NIR_BROAD",
    "NIR_NARROW",
    "SWIR_1",
    "SWIR_2",
]

TM_LAYER_INDICES = [
    2,
    5,
    8,
    11,
]

TM_EMBED_DIM = 768
PYRAMID_CHANNELS = [
    64,
    128,
    256,
    512,
]

PYRAMID_SIZES = [
    128,
    64,
    32,
    16,
]


def make_group_norm(channels):
    for groups in [
        32,
        16,
        8,
        4,
        2,
        1,
    ]:
        if (
            channels
            % groups
            == 0
        ):
            return nn.GroupNorm(
                groups,
                channels,
            )

    raise RuntimeError(
        "No GroupNorm divisor."
    )


class ConvNormAct(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
    ):
        super().__init__()

        padding = (
            kernel_size // 2
        )

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                padding=padding,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class SEBlock(nn.Module):
    def __init__(
        self,
        channels,
        reduction=8,
    ):
        super().__init__()

        hidden = max(
            channels // reduction,
            8,
        )

        self.fc1 = nn.Conv2d(
            channels,
            hidden,
            kernel_size=1,
        )

        self.fc2 = nn.Conv2d(
            hidden,
            channels,
            kernel_size=1,
        )

    def forward(self, x):
        w = F.adaptive_avg_pool2d(
            x,
            1,
        )

        w = F.gelu(
            self.fc1(w)
        )

        w = torch.sigmoid(
            self.fc2(w)
        )

        return x * w


class ConvNeXtRefine(nn.Module):
    def __init__(
        self,
        channels,
        drop=0.0,
    ):
        super().__init__()

        self.dw = nn.Conv2d(
            channels,
            channels,
            kernel_size=7,
            padding=3,
            groups=channels,
        )

        self.norm = make_group_norm(
            channels
        )

        self.pw1 = nn.Conv2d(
            channels,
            channels * 4,
            kernel_size=1,
        )

        self.pw2 = nn.Conv2d(
            channels * 4,
            channels,
            kernel_size=1,
        )

        self.drop = nn.Dropout2d(
            drop
        )

        self.se = SEBlock(
            channels
        )

    def forward(self, x):
        residual = x

        x = self.dw(x)

        x = self.norm(x)

        x = F.gelu(
            self.pw1(x)
        )

        x = self.drop(
            self.pw2(x)
        )

        x = self.se(x)

        return residual + x


class ModalityProjector(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        out_size,
    ):
        super().__init__()

        self.out_size = out_size

        self.proj = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
            ConvNeXtRefine(
                out_channels,
                drop=0.0,
            ),
        )

    def forward(self, x):
        x = self.proj(x)

        if (
            x.shape[-2:]
            != (
                self.out_size,
                self.out_size,
            )
        ):
            x = F.interpolate(
                x,
                size=(
                    self.out_size,
                    self.out_size,
                ),
                mode="bilinear",
                align_corners=False,
            )

        return x


class ReliabilityRouter(nn.Module):
    """
    Spatial SAR-optical router.

    P(clear) is used as a prior on the optical route,
    while the learned router can still override that prior.
    """

    def __init__(
        self,
        channels,
        hidden=64,
        prior_strength=0.50,
    ):
        super().__init__()

        hidden = min(
            hidden,
            max(
                channels // 2,
                16,
            ),
        )

        self.prior_strength = (
            float(
                prior_strength
            )
        )

        self.net = nn.Sequential(
            nn.Conv2d(
                channels * 2 + 1,
                hidden,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(
                hidden
            ),
            nn.GELU(),
            nn.Conv2d(
                hidden,
                2,
                kernel_size=1,
            ),
        )

        # Start near neutral learned routing.
        nn.init.zeros_(
            self.net[-1].weight
        )

        nn.init.zeros_(
            self.net[-1].bias
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        r = F.interpolate(
            pclear,
            size=sar.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        r = r.clamp(
            0.0,
            1.0,
        )

        logits = self.net(
            torch.cat(
                [
                    sar,
                    optical,
                    r,
                ],
                dim=1,
            )
        )

        # Optical prior. Clamp avoids -inf at P(clear)=0.
        logits[
            :,
            1:2,
        ] = (
            logits[
                :,
                1:2,
            ]
            + self.prior_strength
            * torch.log(
                r.clamp(
                    min=0.05,
                    max=1.0,
                )
            )
        )

        weights = torch.softmax(
            logits,
            dim=1,
        )

        w_sar = weights[
            :,
            0:1,
        ]

        w_opt = weights[
            :,
            1:2,
        ]

        fused = (
            w_sar * sar
            + w_opt * optical
        )

        return (
            fused,
            weights,
        )


class DeepRIFT(nn.Module):
    """
    Zero-initialized reliability-conditioned cross attention.

    When P(clear) is high:
      SAR can borrow more optical context.

    When P(clear) is low:
      optical can borrow more SAR context.

    Output projections start at zero, so the block is an
    exact identity at initialization.
    """

    def __init__(
        self,
        channels=512,
        heads=8,
        dropout=0.0,
    ):
        super().__init__()

        self.channels = channels

        self.sar_norm = nn.LayerNorm(
            channels
        )

        self.opt_norm = nn.LayerNorm(
            channels
        )

        self.sar_from_opt = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.opt_from_sar = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.sar_out = nn.Linear(
            channels,
            channels,
        )

        self.opt_out = nn.Linear(
            channels,
            channels,
        )

        nn.init.zeros_(
            self.sar_out.weight
        )

        nn.init.zeros_(
            self.sar_out.bias
        )

        nn.init.zeros_(
            self.opt_out.weight
        )

        nn.init.zeros_(
            self.opt_out.bias
        )

    @staticmethod
    def _flatten(x):
        return (
            x.flatten(2)
            .transpose(1, 2)
            .contiguous()
        )

    @staticmethod
    def _unflatten(
        tokens,
        height,
        width,
    ):
        return (
            tokens
            .transpose(1, 2)
            .reshape(
                tokens.shape[0],
                tokens.shape[2],
                height,
                width,
            )
            .contiguous()
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        b, c, h, w = sar.shape

        sar_t = self._flatten(
            sar
        )

        opt_t = self._flatten(
            optical
        )

        r = F.adaptive_avg_pool2d(
            pclear,
            output_size=(h, w),
        )

        r_t = (
            r.flatten(2)
            .transpose(1, 2)
            .clamp(0.0, 1.0)
        )

        sar_n = self.sar_norm(
            sar_t
        )

        opt_n = self.opt_norm(
            opt_t
        )

        # Clear optical context receives larger K/V magnitude.
        opt_context = (
            opt_n
            * (
                0.25
                + 0.75 * r_t
            )
        )

        sar_delta, _ = (
            self.sar_from_opt(
                query=sar_n,
                key=opt_context,
                value=opt_context,
                need_weights=False,
            )
        )

        # Under low P(clear), the optical branch borrows SAR more.
        opt_delta, _ = (
            self.opt_from_sar(
                query=opt_n,
                key=sar_n,
                value=sar_n,
                need_weights=False,
            )
        )

        sar_t = (
            sar_t
            + r_t
            * self.sar_out(
                sar_delta
            )
        )

        opt_t = (
            opt_t
            + (1.0 - r_t)
            * self.opt_out(
                opt_delta
            )
        )

        sar = self._unflatten(
            sar_t,
            h,
            w,
        )

        optical = self._unflatten(
            opt_t,
            h,
            w,
        )

        return sar, optical


class DecoderBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
    ):
        super().__init__()

        self.fuse = nn.Sequential(
            ConvNormAct(
                in_channels
                + skip_channels,
                out_channels,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
        )

    def forward(
        self,
        x,
        skip,
    ):
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        x = torch.cat(
            [
                x,
                skip,
            ],
            dim=1,
        )

        return self.fuse(x)


class FloodShiftTerraMindRIFT(nn.Module):
    def __init__(
        self,
        num_classes=3,
        pretrained_backbone=False,
    ):
        super().__init__()

        # concat is used deliberately:
        # each spatial token contains S2 and S1 embeddings
        # concatenated in the modality order supplied below.
        self.backbone = (
            BACKBONE_REGISTRY.build(
                "terramind_v1_base",
                pretrained=pretrained_backbone,
                img_size=TILE_SIZE,
                modalities=[
                    "S2L1C",
                    "S1GRD",
                ],
                bands={
                    "S2L1C":
                        TM_OPTICAL_BAND_NAMES,
                },
                merge_method="concat",
            )
        )

        self.register_buffer(
            "project_opt_mean",
            torch.tensor(
                OPTICAL_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_opt_std",
            torch.tensor(
                OPTICAL_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_mean",
            torch.tensor(
                SAR_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_std",
            torch.tensor(
                SAR_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_mean",
            torch.tensor(
                TM_S2_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_std",
            torch.tensor(
                TM_S2_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_mean",
            torch.tensor(
                TM_S1_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_std",
            torch.tensor(
                TM_S1_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.sar_projectors = (
            nn.ModuleList()
        )

        self.opt_projectors = (
            nn.ModuleList()
        )

        self.routers = nn.ModuleList()

        for (
            channels,
            out_size,
        ) in zip(
            PYRAMID_CHANNELS,
            PYRAMID_SIZES,
        ):
            self.sar_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.opt_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.routers.append(
                ReliabilityRouter(
                    channels,
                    prior_strength=0.50,
                )
            )

        self.deep_rift = DeepRIFT(
            channels=PYRAMID_CHANNELS[-1],
            heads=8,
            dropout=0.0,
        )

        self.deep_context = nn.Sequential(
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
        )

        self.decoder3 = DecoderBlock(
            512,
            256,
            256,
        )

        self.decoder2 = DecoderBlock(
            256,
            128,
            128,
        )

        self.decoder1 = DecoderBlock(
            128,
            64,
            64,
        )

        self.final_refine = nn.Sequential(
            ConvNormAct(
                64,
                64,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                64,
                drop=0.05,
            ),
        )

        self.segmentation_head = (
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            )
        )

        self.aux_head = nn.Sequential(
            nn.Conv2d(
                128,
                64,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(64),
            nn.GELU(),
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            ),
        )

    def to_terramind_inputs(
        self,
        sar_z,
        optical_z,
    ):
        # Detect frozen invalid-fill pixels BEFORE inverse affine.
        opt_invalid = (
            optical_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        sar_invalid = (
            sar_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        # Frozen optical z-score -> reflectance -> DN.
        optical_reflectance = (
            optical_z
            * self.project_opt_std
            + self.project_opt_mean
        )

        optical_dn = (
            optical_reflectance
            * 10000.0
        )

        optical_tm = (
            optical_dn
            - self.tm_opt_mean
        ) / self.tm_opt_std

        # Frozen SAR z-score -> raw dB.
        sar_db = (
            sar_z
            * self.project_sar_std
            + self.project_sar_mean
        )

        sar_tm = (
            sar_db
            - self.tm_sar_mean
        ) / self.tm_sar_std

        # Preserve invalid-fill convention in model space.
        optical_tm = optical_tm.masked_fill(
            opt_invalid,
            0.0,
        )

        sar_tm = sar_tm.masked_fill(
            sar_invalid,
            0.0,
        )

        return (
            sar_tm,
            optical_tm,
        )

    @staticmethod
    def tokens_to_map(tokens):
        b, n, c = tokens.shape

        side = int(
            math.isqrt(n)
        )

        if (
            side * side
            != n
        ):
            raise RuntimeError(
                f"TerraMind token count {n} "
                "is not a square grid."
            )

        return (
            tokens
            .transpose(1, 2)
            .reshape(
                b,
                c,
                side,
                side,
            )
            .contiguous()
        )

    def extract_modality_tokens(
        self,
        tm_outputs,
    ):
        if (
            not isinstance(
                tm_outputs,
                (list, tuple),
            )
            or len(tm_outputs) < 12
        ):
            raise RuntimeError(
                "Unexpected TerraMind output. "
                "Expected list of 12 transformer-layer outputs."
            )

        sar_tokens = []
        opt_tokens = []

        for layer_index in TM_LAYER_INDICES:
            tokens = tm_outputs[
                layer_index
            ]

            if (
                not torch.is_tensor(tokens)
                or tokens.ndim != 3
            ):
                raise RuntimeError(
                    "TerraMind concat output should be "
                    "[B, tokens, embedding]."
                )

            expected_dim = (
                TM_EMBED_DIM * 2
            )

            if (
                tokens.shape[-1]
                != expected_dim
            ):
                raise RuntimeError(
                    "Expected concat embedding dimension "
                    f"{expected_dim}, got "
                    f"{tokens.shape[-1]}. "
                    "Check TerraTorch version and merge_method."
                )

            # modalities were supplied in order:
            # [S2L1C, S1GRD]
            optical_part = tokens[
                :,
                :,
                :TM_EMBED_DIM,
            ]

            sar_part = tokens[
                :,
                :,
                TM_EMBED_DIM:,
            ]

            opt_tokens.append(
                optical_part
            )

            sar_tokens.append(
                sar_part
            )

        return (
            sar_tokens,
            opt_tokens,
        )

    def forward(
        self,
        sar,
        optical,
        reliability,
        return_aux=False,
    ):
        (
            sar_tm,
            optical_tm,
        ) = self.to_terramind_inputs(
            sar,
            optical,
        )

        tm_outputs = self.backbone(
            {
                "S2L1C": optical_tm,
                "S1GRD": sar_tm,
            }
        )

        (
            sar_tokens,
            opt_tokens,
        ) = self.extract_modality_tokens(
            tm_outputs
        )

        sar_maps = []
        opt_maps = []

        for i in range(4):
            sar_map = (
                self.tokens_to_map(
                    sar_tokens[i]
                )
            )

            opt_map = (
                self.tokens_to_map(
                    opt_tokens[i]
                )
            )

            sar_map = (
                self.sar_projectors[i](
                    sar_map
                )
            )

            opt_map = (
                self.opt_projectors[i](
                    opt_map
                )
            )

            sar_maps.append(
                sar_map
            )

            opt_maps.append(
                opt_map
            )

        # RIFT only at deepest 16x16 level.
        (
            sar_maps[-1],
            opt_maps[-1],
        ) = self.deep_rift(
            sar_maps[-1],
            opt_maps[-1],
            reliability,
        )

        fused = []
        router_weights = []

        for i in range(4):
            (
                fused_i,
                weights_i,
            ) = self.routers[i](
                sar_maps[i],
                opt_maps[i],
                reliability,
            )

            fused.append(
                fused_i
            )

            router_weights.append(
                weights_i
            )

        f0, f1, f2, f3 = fused

        x = self.deep_context(
            f3
        )

        x = self.decoder3(
            x,
            f2,
        )

        x = self.decoder2(
            x,
            f1,
        )

        aux_logits = self.aux_head(
            x
        )

        x = self.decoder1(
            x,
            f0,
        )

        x = F.interpolate(
            x,
            size=(
                TILE_SIZE,
                TILE_SIZE,
            ),
            mode="bilinear",
            align_corners=False,
        )

        x = self.final_refine(
            x
        )

        logits = (
            self.segmentation_head(
                x
            )
        )

        if return_aux:
            return {
                "logits": logits,
                "aux_logits": aux_logits,
                "router_weights": router_weights,
            }

        return logits

In [15]:
# ============================================================
# CELL: P15-C07 — REBUILD EXACT ACCEPTED 04H-CAL MODEL
# ============================================================
class HierarchicalResidualHead(nn.Module):
    def __init__(self, in_channels=16, hidden=48, max_delta=1.50):
        super().__init__()
        self.max_delta = float(max_delta)
        self.net = nn.Sequential(
            ConvNormAct(in_channels, hidden, 3),
            ConvNeXtRefine(hidden, drop=0.05),
            ConvNormAct(hidden, hidden, 3),
            nn.Conv2d(hidden, 2, kernel_size=1),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, base_prob, sar, optical, reliability):
        x = torch.cat([base_prob, sar, optical, reliability], dim=1)
        raw = self.net(x).float()
        return self.max_delta * torch.tanh(raw / self.max_delta)

class FloodShift04H(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.head = HierarchicalResidualHead()

        for p in self.base.parameters():
            p.requires_grad = False
        self.base.eval()

    @property
    def backbone(self):
        return self.base.backbone

    def train(self, mode=True):
        super().train(mode)
        self.base.eval()
        self.head.train(mode)
        return self

    def forward(self, sar, optical, reliability, return_aux=False):
        with torch.no_grad():
            base_out = self.base(
                sar,
                optical,
                reliability,
                return_aux=True,
            )
            base_logits = base_out["logits"].float()
            base_prob = torch.softmax(base_logits, dim=1)

        correction = self.head(
            base_prob.to(sar.dtype),
            sar,
            optical,
            reliability,
        )

        delta_water = correction[:, 0]
        delta_type = correction[:, 1]

        base_water_logit = (
            torch.logsumexp(base_logits[:, 1:3], dim=1)
            - base_logits[:, 0]
        )
        base_type_logit = base_logits[:, 2] - base_logits[:, 1]

        water_logit = base_water_logit + delta_water
        type_logit = base_type_logit + delta_type

        p_water = torch.sigmoid(water_logit)
        p_temp_given_water = torch.sigmoid(type_logit)

        p0 = 1.0 - p_water
        p1 = p_water * (1.0 - p_temp_given_water)
        p2 = p_water * p_temp_given_water

        probs = torch.stack([p0, p1, p2], dim=1).clamp_min(1e-7)
        log_probs = probs.log()

        if return_aux:
            return {
                "logits": log_probs,
                "probs": probs,
                "base_logits": base_logits,
                "base_prob": base_prob,
                "base_water_logit": base_water_logit,
                "base_type_logit": base_type_logit,
                "water_logit": water_logit,
                "type_logit": type_logit,
                "delta_water": delta_water,
                "delta_type": delta_type,
            }

        return log_probs

payload = torch.load(CHECKPOINT_04H, map_location="cpu", weights_only=False)

assert payload.get("base_sha256") == EXPECTED_04E_SHA256
assert int(payload.get("epoch", -1)) == 4
assert payload.get("tag") == "best_mIoU"

base = FloodShiftTerraMindRIFT(
    num_classes=NUM_CLASSES,
    pretrained_backbone=False,
)
model = FloodShift04H(base)

missing, unexpected = model.load_state_dict(
    payload["state_dict"],
    strict=False,
)

assert not missing, f"Missing keys: {missing[:10]}"
assert not unexpected, f"Unexpected keys: {unexpected[:10]}"

bad = [
    n for n, t in model.state_dict().items()
    if torch.is_floating_point(t) and not torch.isfinite(t).all()
]
assert not bad

model = model.to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad = False

print("=" * 112)
print("P15-C07 — ACCEPTED 04H-CAL MODEL READY")
print("=" * 112)
print("Epoch          :", payload["epoch"])
print("Tag            :", payload["tag"])
print("Type bias      :", TYPE_BIAS)
print("Temperature    :", FROZEN_TEMPERATURE)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("P15-C07 COMPLETE ✅")


INFO:root:Loaded weights for BLUE in position 0 of patch embed
INFO:root:Loaded weights for GREEN in position 1 of patch embed
INFO:root:Loaded weights for RED in position 2 of patch embed
INFO:root:Loaded weights for RED_EDGE_1 in position 3 of patch embed
INFO:root:Loaded weights for RED_EDGE_2 in position 4 of patch embed
INFO:root:Loaded weights for RED_EDGE_3 in position 5 of patch embed
INFO:root:Loaded weights for NIR_BROAD in position 6 of patch embed
INFO:root:Loaded weights for NIR_NARROW in position 7 of patch embed
INFO:root:Loaded weights for SWIR_1 in position 8 of patch embed
INFO:root:Loaded weights for SWIR_2 in position 9 of patch embed


P15-C07 — ACCEPTED 04H-CAL MODEL READY
Epoch          : 4
Tag            : best_mIoU
Type bias      : -0.15
Temperature    : 0.989592432975769
Trainable params: 0
P15-C07 COMPLETE ✅


In [14]:
# ============================================================
# PATCH — NOTEBOOK 07 TEMPERATURE NAME COMPATIBILITY
# ============================================================

FROZEN_TEMPERATURE = TEMPERATURE

assert abs(
    float(FROZEN_TEMPERATURE) - float(TEMPERATURE)
) < 1e-12

print("✅ Temperature alias fixed")
print("TEMPERATURE        :", TEMPERATURE)
print("FROZEN_TEMPERATURE:", FROZEN_TEMPERATURE)

✅ Temperature alias fixed
TEMPERATURE        : 0.989592432975769
FROZEN_TEMPERATURE: 0.989592432975769


In [16]:
# ============================================================
# CELL: P15-C08 — METRICS + FROZEN 4-WAY TTA + COUNTERFACTUALS
# ============================================================
TTA_MODES = ["identity", "hflip", "vflip", "rot180"]

def confusion_from_predictions(pred, target):
    valid = target != IGNORE_INDEX
    target_v = target[valid].long()
    pred_v = pred[valid].long()

    indices = NUM_CLASSES * target_v + pred_v
    return torch.bincount(
        indices,
        minlength=NUM_CLASSES * NUM_CLASSES,
    ).reshape(NUM_CLASSES, NUM_CLASSES)

def metrics_from_confusion(cm):
    if torch.is_tensor(cm):
        cm = cm.detach().cpu().numpy().astype(np.float64)
    else:
        cm = np.asarray(cm, dtype=np.float64)

    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp

    den_iou = tp + fp + fn
    iou = np.divide(
        tp,
        den_iou,
        out=np.full_like(tp, np.nan),
        where=den_iou > 0,
    )

    accuracy = float(tp.sum() / max(cm.sum(), 1.0))

    out = {
        "mIoU": float(np.nanmean(iou)),
        "accuracy": accuracy,
        "confusion": cm.tolist(),
    }

    for i, name in enumerate(CLASS_NAMES):
        out[f"{name}_IoU"] = float(iou[i])

    return out

def compact_metrics(m):
    return {
        "mIoU": round(m["mIoU"], 4),
        "non_flood": round(m["non_flood_IoU"], 4),
        "permanent": round(m["permanent_water_IoU"], 4),
        "temporary": round(m["temporary_flood_IoU"], 4),
    }

def _tta_transform(x, mode):
    if mode == "identity":
        return x
    if mode == "hflip":
        return torch.flip(x, dims=[-1])
    if mode == "vflip":
        return torch.flip(x, dims=[-2])
    if mode == "rot180":
        return torch.rot90(x, k=2, dims=(-2, -1))
    raise ValueError(mode)

def _tta_inverse(x, mode):
    return _tta_transform(x, mode)

def calibrated_log_probs_from_aux(aux):
    water_logit = aux["water_logit"].float()
    type_logit = aux["type_logit"].float() + TYPE_BIAS

    p_water = torch.sigmoid(water_logit)
    p_temp_given_water = torch.sigmoid(type_logit)

    p0 = 1.0 - p_water
    p1 = p_water * (1.0 - p_temp_given_water)
    p2 = p_water * p_temp_given_water

    probs = torch.stack([p0, p1, p2], dim=1).clamp_min(1e-7)
    return probs.log()

@torch.inference_mode()
def tta_log_probs_variant(model_to_eval, sar, optical, reliability, variant):
    if variant == "full":
        sar_v = sar
        opt_v = optical
        rel_v = reliability

    elif variant == "no_sar":
        sar_v = torch.zeros_like(sar)
        opt_v = optical
        rel_v = reliability

    elif variant == "no_optical":
        sar_v = sar
        opt_v = torch.zeros_like(optical)
        rel_v = torch.zeros_like(reliability)

    else:
        raise ValueError(variant)

    total = None

    for mode in TTA_MODES:
        sar_t = _tta_transform(sar_v, mode)
        opt_t = _tta_transform(opt_v, mode)
        rel_t = _tta_transform(rel_v, mode)

        with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
            aux = model_to_eval(
                sar_t,
                opt_t,
                rel_t,
                return_aux=True,
            )

        lp = calibrated_log_probs_from_aux(aux)
        lp = _tta_inverse(lp, mode).float()

        total = lp if total is None else total + lp

    return total / len(TTA_MODES)

def probs_with_temperature(log_probs):
    return torch.softmax(
        log_probs.float() / FROZEN_TEMPERATURE,
        dim=1,
    )

def normalized_entropy(probs):
    return (
        -(probs * probs.clamp_min(1e-8).log()).sum(dim=1)
        / math.log(NUM_CLASSES)
    )

def normalized_js(p, q):
    """
    Jensen-Shannon divergence normalized by ln(2), so output is in [0,1].
    Shapes: [B,C,H,W] -> [B,H,W]
    """
    p = p.clamp_min(1e-8)
    q = q.clamp_min(1e-8)
    m = 0.5 * (p + q)

    kl_pm = (p * (p.log() - m.log())).sum(dim=1)
    kl_qm = (q * (q.log() - m.log())).sum(dim=1)

    js = 0.5 * (kl_pm + kl_qm)
    return (js / math.log(2.0)).clamp(0.0, 1.0)

print("P15-C08 COMPLETE ✅")


P15-C08 COMPLETE ✅


In [19]:
# ============================================================
# CELL: P15-C09 — FROZEN INFERENCE + SCENE MAP STITCHING
#
# Packet-facing maps contain NO ground truth.
# Ground truth is touched only for the integrity accumulators below.
# ============================================================
SCENE_H = 512
SCENE_W = 512

def init_scene_maps():
    return {
        "prediction": np.zeros((SCENE_H, SCENE_W), dtype=np.uint8),
        "confidence": np.zeros((SCENE_H, SCENE_W), dtype=np.float32),
        "entropy": np.zeros((SCENE_H, SCENE_W), dtype=np.float32),
        "pclear": np.zeros((SCENE_H, SCENE_W), dtype=np.float32),
        "js_sar": np.zeros((SCENE_H, SCENE_W), dtype=np.float32),
        "js_optical": np.zeros((SCENE_H, SCENE_W), dtype=np.float32),
        "flip_no_sar": np.zeros((SCENE_H, SCENE_W), dtype=np.uint8),
        "flip_no_optical": np.zeros((SCENE_H, SCENE_W), dtype=np.uint8),
        "coverage": np.zeros((SCENE_H, SCENE_W), dtype=np.uint8),
        "sar_vv_db": np.zeros((SCENE_H, SCENE_W), dtype=np.float32),
        "rgb_reflectance": np.zeros((SCENE_H, SCENE_W, 3), dtype=np.float32),
    }

scene_maps = {}

cm_full = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)
cm_no_sar = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)
cm_no_opt = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)

audit_js_sar_sum = 0.0
audit_js_opt_sum = 0.0
audit_valid_n = 0

model.eval()

with torch.inference_mode():
    for bi, batch in enumerate(val_attr_loader, start=1):
        sar = batch["sar"].to(DEVICE, non_blocking=True)
        opt = batch["optical"].to(DEVICE, non_blocking=True)
        rel = batch["reliability"].to(DEVICE, non_blocking=True)

        # AUDIT ONLY. Never exported to a packet.
        target = batch["target"].to(DEVICE, non_blocking=True)

        lp_full = tta_log_probs_variant(model, sar, opt, rel, "full")
        lp_no_sar = tta_log_probs_variant(model, sar, opt, rel, "no_sar")
        lp_no_opt = tta_log_probs_variant(model, sar, opt, rel, "no_optical")

        p_full = probs_with_temperature(lp_full)
        p_no_sar = probs_with_temperature(lp_no_sar)
        p_no_opt = probs_with_temperature(lp_no_opt)

        pred_full = p_full.argmax(1)
        pred_no_sar = p_no_sar.argmax(1)
        pred_no_opt = p_no_opt.argmax(1)

        entropy = normalized_entropy(p_full)
        confidence = p_full.max(dim=1).values
        js_sar = normalized_js(p_full, p_no_sar)
        js_opt = normalized_js(p_full, p_no_opt)

        # ------------------------------
        # Isolated Notebook-06 audit.
        # ------------------------------
        cm_full += confusion_from_predictions(pred_full, target).cpu()
        cm_no_sar += confusion_from_predictions(pred_no_sar, target).cpu()
        cm_no_opt += confusion_from_predictions(pred_no_opt, target).cpu()

        audit_valid = target != IGNORE_INDEX
        audit_js_sar_sum += float(js_sar[audit_valid].sum().cpu())
        audit_js_opt_sum += float(js_opt[audit_valid].sum().cpu())
        audit_valid_n += int(audit_valid.sum().cpu())

        # ------------------------------
        # Deployment-style coverage.
        # Uses sensors only, not target.
        # ------------------------------
        sar_present = sar.abs().sum(dim=1) > 1e-12
        opt_present = opt.abs().sum(dim=1) > 1e-12
        coverage = sar_present | opt_present

        scene_ids = list(batch["scene_id"])
        tile_indices = batch["tile_index"]

        for i in range(sar.shape[0]):
            sid = str(scene_ids[i])
            tid = int(tile_indices[i])

            if sid not in scene_maps:
                scene_maps[sid] = init_scene_maps()

            y0, x0 = TILE_OFFSETS[tid]
            ys = slice(y0, y0 + TILE_SIZE)
            xs = slice(x0, x0 + TILE_SIZE)

            pred_np = pred_full[i].detach().cpu().numpy().astype(np.uint8)
            conf_np = confidence[i].detach().cpu().numpy().astype(np.float32)
            ent_np = entropy[i].detach().cpu().numpy().astype(np.float32)
            rel_np = rel[i, 0].detach().cpu().numpy().astype(np.float32)
            js_s_np = js_sar[i].detach().cpu().numpy().astype(np.float32)
            js_o_np = js_opt[i].detach().cpu().numpy().astype(np.float32)

            flip_s_np = (
                pred_no_sar[i] != pred_full[i]
            ).detach().cpu().numpy().astype(np.uint8)

            flip_o_np = (
                pred_no_opt[i] != pred_full[i]
            ).detach().cpu().numpy().astype(np.uint8)

            cov_np = coverage[i].detach().cpu().numpy().astype(np.uint8)

            sar_np = sar[i].detach().cpu().numpy().astype(np.float32)
            opt_np = opt[i].detach().cpu().numpy().astype(np.float32)

            # Reverse frozen normalization only for visualization/evidence rendering.
            vv_db = sar_np[0] * float(SAR_STD[0]) + float(SAR_MEAN[0])

            optical_ref = (
                opt_np
                * OPTICAL_STD[:, None, None]
                + OPTICAL_MEAN[:, None, None]
            )

            # Optical10 order begins B2,B3,B4.
            rgb = np.stack(
                [
                    optical_ref[2],  # B4
                    optical_ref[1],  # B3
                    optical_ref[0],  # B2
                ],
                axis=-1,
            ).astype(np.float32)

            sm = scene_maps[sid]
            sm["prediction"][ys, xs] = pred_np
            sm["confidence"][ys, xs] = conf_np
            sm["entropy"][ys, xs] = ent_np
            sm["pclear"][ys, xs] = rel_np
            sm["js_sar"][ys, xs] = js_s_np
            sm["js_optical"][ys, xs] = js_o_np
            sm["flip_no_sar"][ys, xs] = flip_s_np
            sm["flip_no_optical"][ys, xs] = flip_o_np
            sm["coverage"][ys, xs] = cov_np
            sm["sar_vv_db"][ys, xs] = vv_db
            sm["rgb_reflectance"][ys, xs] = rgb

        if bi == 1 or bi % 20 == 0 or bi == len(val_attr_loader):
            print(f"[{bi:03d}/{len(val_attr_loader)}] frozen GeoEngine inference")

assert len(scene_maps) == 34

print("Scenes stitched:", len(scene_maps))
print("P15-C09 COMPLETE ✅")


[001/68] frozen GeoEngine inference
[020/68] frozen GeoEngine inference
[040/68] frozen GeoEngine inference
[060/68] frozen GeoEngine inference
[068/68] frozen GeoEngine inference
Scenes stitched: 34
P15-C09 COMPLETE ✅


In [20]:
# ============================================================
# CELL: P15-C10 — MANDATORY NOTEBOOK-06 INTEGRITY AUDIT
# ============================================================
audit_full = metrics_from_confusion(cm_full)
audit_no_sar = metrics_from_confusion(cm_no_sar)
audit_no_opt = metrics_from_confusion(cm_no_opt)

expected_integrity = protocol06["counterfactual_integrity"]
expected_summary = protocol06["attribution_summary"]

audit_mean_js_sar = audit_js_sar_sum / max(audit_valid_n, 1)
audit_mean_js_opt = audit_js_opt_sum / max(audit_valid_n, 1)

print("=" * 112)
print("P15-C10 — NOTEBOOK-06 INTEGRITY AUDIT")
print("=" * 112)

print("FULL      :", compact_metrics(audit_full))
print("NO SAR    :", compact_metrics(audit_no_sar))
print("NO OPTICAL:", compact_metrics(audit_no_opt))
print()
print("Audit mean JS SAR    :", audit_mean_js_sar)
print("Notebook-06 JS SAR   :", expected_summary["mean_js_sar"])
print("Audit mean JS optical:", audit_mean_js_opt)
print("Notebook-06 JS opt   :", expected_summary["mean_js_optical"])

for key, current in [
    ("full", audit_full),
    ("no_sar", audit_no_sar),
    ("no_optical", audit_no_opt),
]:
    expected = expected_integrity[key]
    assert abs(current["mIoU"] - expected["mIoU"]) < 0.0020

assert abs(
    audit_mean_js_sar
    - float(expected_summary["mean_js_sar"])
) < 1e-4

assert abs(
    audit_mean_js_opt
    - float(expected_summary["mean_js_optical"])
) < 1e-4

print()
print("✅ accepted 04H-CAL path reproduced")
print("✅ counterfactual paths reproduced")
print("✅ attribution means reproduced")
print("✅ audit target is NOT written into any packet")
print("P15-C10 COMPLETE ✅")


P15-C10 — NOTEBOOK-06 INTEGRITY AUDIT
FULL      : {'mIoU': 0.8265, 'non_flood': 0.9706, 'permanent': 0.8885, 'temporary': 0.6204}
NO SAR    : {'mIoU': 0.5609, 'non_flood': 0.9512, 'permanent': 0.3026, 'temporary': 0.4288}
NO OPTICAL: {'mIoU': 0.2917, 'non_flood': 0.8602, 'permanent': 0.0148, 'temporary': 0.0}

Audit mean JS SAR    : 0.0428984161748137
Notebook-06 JS SAR   : 0.04289841483544225
Audit mean JS optical: 0.13324325941456075
Notebook-06 JS opt   : 0.1332432571420121

✅ accepted 04H-CAL path reproduced
✅ counterfactual paths reproduced
✅ attribution means reproduced
✅ audit target is NOT written into any packet
P15-C10 COMPLETE ✅


In [21]:
# ============================================================
# CELL: P15-C11 — DETERMINISTIC GEOSPATIAL HELPERS
# ============================================================
LOW_PCLEAR_THRESHOLD = 0.50
PACKET_SCHEMA_VERSION = "FloodShift-GeoEvidence-v1"

def finite_or_none(x):
    if x is None:
        return None
    x = float(x)
    return x if np.isfinite(x) else None

def quantile_or_none(x, q):
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return None
    return float(np.quantile(x, q))

def pixel_area_map_km2(transform, crs, height, width):
    """
    Deterministic area map.

    Projected CRS:
      affine cell area × CRS linear-unit conversion.

    Geographic CRS:
      row-wise geodesic cell area when grid has no rotation.

    Returns:
      area_map_km2 or None,
      method string.
    """
    if crs is None or not PYPROJ_AVAILABLE:
        return None, "unavailable"

    crs_obj = CRS.from_user_input(crs)

    if crs_obj.is_projected:
        det = abs(
            transform.a * transform.e
            - transform.b * transform.d
        )

        axis = crs_obj.axis_info
        fx = (
            float(axis[0].unit_conversion_factor)
            if len(axis) >= 1 and axis[0].unit_conversion_factor
            else 1.0
        )
        fy = (
            float(axis[1].unit_conversion_factor)
            if len(axis) >= 2 and axis[1].unit_conversion_factor
            else fx
        )

        area_km2 = det * fx * fy / 1_000_000.0

        return (
            np.full(
                (height, width),
                area_km2,
                dtype=np.float64,
            ),
            "projected_affine_exact",
        )

    if crs_obj.is_geographic:
        # Normal north-up geographic raster.
        if abs(transform.b) > 1e-12 or abs(transform.d) > 1e-12:
            return None, "geographic_rotated_grid_unsupported"

        geod = crs_obj.get_geod()
        out = np.zeros((height, width), dtype=np.float64)

        x0 = float(transform.c)
        x1 = float(transform.c + transform.a)

        for row in range(height):
            y0 = float(transform.f + row * transform.e)
            y1 = float(transform.f + (row + 1) * transform.e)

            area_m2, _ = geod.polygon_area_perimeter(
                [x0, x1, x1, x0],
                [y0, y0, y1, y1],
            )

            out[row, :] = abs(area_m2) / 1_000_000.0

        return out, "geographic_geodesic_row_exact"

    return None, "unsupported_crs"

def centroid_wgs84(mask, transform, crs):
    rows, cols = np.nonzero(mask)
    if len(rows) == 0:
        return None

    r = float(rows.mean())
    c = float(cols.mean())

    x, y = rasterio.transform.xy(
        transform,
        r,
        c,
        offset="center",
    )

    if crs is None:
        return {
            "row": r,
            "col": c,
            "longitude": None,
            "latitude": None,
        }

    try:
        lon, lat = rio_transform(
            crs,
            "EPSG:4326",
            [x],
            [y],
        )

        return {
            "row": r,
            "col": c,
            "longitude": float(lon[0]),
            "latitude": float(lat[0]),
        }

    except Exception:
        return {
            "row": r,
            "col": c,
            "longitude": None,
            "latitude": None,
        }

def component_geometry(temp_mask, area_map, transform, crs):
    structure = np.ones((3, 3), dtype=np.uint8)
    labels, n_components = ndimage.label(
        temp_mask.astype(bool),
        structure=structure,
    )

    if n_components == 0:
        return {
            "component_count": 0,
            "largest_component_pixels": 0,
            "largest_component_area_km2": 0.0 if area_map is not None else None,
            "largest_component_fraction_of_temp": 0.0,
            "largest_component_centroid": None,
        }

    total_temp = int(temp_mask.sum())

    best_label = None
    best_pixels = -1
    best_area = -1.0

    for lab in range(1, n_components + 1):
        m = labels == lab
        pixels = int(m.sum())

        if area_map is not None:
            area = float(area_map[m].sum())
            score = area
        else:
            area = None
            score = float(pixels)

        if score > best_area:
            best_area = score
            best_label = lab
            best_pixels = pixels
            best_area_km2 = area

    best_mask = labels == best_label

    return {
        "component_count": int(n_components),
        "largest_component_pixels": int(best_pixels),
        "largest_component_area_km2": finite_or_none(best_area_km2),
        "largest_component_fraction_of_temp": (
            float(best_pixels / total_temp)
            if total_temp > 0
            else 0.0
        ),
        "largest_component_centroid": centroid_wgs84(
            best_mask,
            transform,
            crs,
        ),
    }

def dominance_label(sar_share, optical_share):
    if sar_share >= DOMINANCE_SHARE:
        return "SAR-dominant"
    if optical_share >= DOMINANCE_SHARE:
        return "Optical-dominant"
    return "Balanced"

print("P15-C11 COMPLETE ✅")


P15-C11 COMPLETE ✅


In [22]:
# ============================================================
# CELL: P15-C12 — BUILD GROUNDED EVIDENCE PACKETS
#
# NO ground truth is read here.
# ============================================================
packets = []
summary_rows = []

for scene_id in sorted(scene_maps):
    sm = scene_maps[scene_id]
    row = scene_row_lookup[scene_id]
    paths = resolve_scene_files(row)

    with rasterio.open(paths["sar"]) as src:
        scene_transform = src.transform
        scene_crs = src.crs
        source_width = src.width
        source_height = src.height

    assert source_width >= SCENE_W
    assert source_height >= SCENE_H

    area_map, area_method = pixel_area_map_km2(
        scene_transform,
        scene_crs,
        SCENE_H,
        SCENE_W,
    )

    coverage = sm["coverage"].astype(bool)
    pred = sm["prediction"]

    n_coverage = int(coverage.sum())
    assert n_coverage > 0

    nonflood = coverage & (pred == 0)
    permanent = coverage & (pred == 1)
    temporary = coverage & (pred == 2)
    water = permanent | temporary

    entropy = sm["entropy"]
    pclear = sm["pclear"]
    js_sar = sm["js_sar"]
    js_opt = sm["js_optical"]
    fragility = np.maximum(js_sar, js_opt)

    mean_js_sar = float(js_sar[coverage].mean())
    mean_js_opt = float(js_opt[coverage].mean())
    denom = mean_js_sar + mean_js_opt

    if denom > 1e-12:
        sar_share = mean_js_sar / denom
        optical_share = mean_js_opt / denom
    else:
        sar_share = 0.5
        optical_share = 0.5

    # Frozen Notebook-06 threshold is a TILE-level threshold.
    tile_fragility_means = []
    for tid, (y0, x0) in enumerate(TILE_OFFSETS):
        ys = slice(y0, y0 + TILE_SIZE)
        xs = slice(x0, x0 + TILE_SIZE)
        cov_t = coverage[ys, xs]
        frag_t = fragility[ys, xs]

        if cov_t.any():
            tile_fragility_means.append(
                float(frag_t[cov_t].mean())
            )

    high_fragility_tile_fraction = float(
        np.mean(
            np.asarray(tile_fragility_means)
            >= HIGH_FRAGILITY_THRESHOLD
        )
    )

    if area_map is not None:
        coverage_area = float(area_map[coverage].sum())
        nonflood_area = float(area_map[nonflood].sum())
        permanent_area = float(area_map[permanent].sum())
        temporary_area = float(area_map[temporary].sum())
        water_area = float(area_map[water].sum())
    else:
        coverage_area = None
        nonflood_area = None
        permanent_area = None
        temporary_area = None
        water_area = None

    temp_geom = component_geometry(
        temporary,
        area_map,
        scene_transform,
        scene_crs,
    )

    def masked_mean(arr, mask):
        return (
            float(arr[mask].mean())
            if mask.any()
            else None
        )

    packet = {
        "schema_version": PACKET_SCHEMA_VERSION,
        "scene_id": scene_id,

        "provenance": {
            "segmenter": "FloodShift-04H-CAL",
            "source_04h_sha256": SHA_04H,
            "source_04e_sha256": EXPECTED_04E_SHA256,
            "type_bias": TYPE_BIAS,
            "temperature": TEMPERATURE,
            "uncertainty_source": "FloodShift-05",
            "attribution_source": "FloodShift-06",
            "test_status": "LOCKED_NOT_EVALUATED",
        },

        "geospatial_reference": {
            "crs": str(scene_crs) if scene_crs is not None else None,
            "raster_width": int(SCENE_W),
            "raster_height": int(SCENE_H),
            "area_method": area_method,
        },

        "coverage": {
            "covered_pixels": n_coverage,
            "total_grid_pixels": int(SCENE_H * SCENE_W),
            "coverage_fraction": float(n_coverage / (SCENE_H * SCENE_W)),
            "coverage_area_km2": finite_or_none(coverage_area),
        },

        "segmentation": {
            "non_flood": {
                "pixels": int(nonflood.sum()),
                "fraction_of_coverage": float(nonflood.sum() / n_coverage),
                "area_km2": finite_or_none(nonflood_area),
            },
            "permanent_water": {
                "pixels": int(permanent.sum()),
                "fraction_of_coverage": float(permanent.sum() / n_coverage),
                "area_km2": finite_or_none(permanent_area),
            },
            "temporary_flood": {
                "pixels": int(temporary.sum()),
                "fraction_of_coverage": float(temporary.sum() / n_coverage),
                "area_km2": finite_or_none(temporary_area),
            },
            "predicted_water_total": {
                "pixels": int(water.sum()),
                "fraction_of_coverage": float(water.sum() / n_coverage),
                "area_km2": finite_or_none(water_area),
            },
        },

        "uncertainty": {
            "score": "normalized_predictive_entropy",
            "mean_entropy": float(entropy[coverage].mean()),
            "median_entropy": float(np.median(entropy[coverage])),
            "p90_entropy": quantile_or_none(entropy[coverage], 0.90),
            "review10_threshold": ENTROPY_THRESHOLD_REVIEW_10,
            "review20_threshold": ENTROPY_THRESHOLD_REVIEW_20,
            "review10_fraction": float(
                np.mean(
                    entropy[coverage]
                    > ENTROPY_THRESHOLD_REVIEW_10
                )
            ),
            "review20_fraction": float(
                np.mean(
                    entropy[coverage]
                    > ENTROPY_THRESHOLD_REVIEW_20
                )
            ),
            "temporary_flood_mean_entropy": masked_mean(
                entropy,
                temporary,
            ),
        },

        "sensor_quality": {
            "optical_reliability_definition": "P(clear)",
            "mean_pclear": float(pclear[coverage].mean()),
            "median_pclear": float(np.median(pclear[coverage])),
            "low_pclear_threshold": LOW_PCLEAR_THRESHOLD,
            "low_pclear_fraction": float(
                np.mean(
                    pclear[coverage]
                    < LOW_PCLEAR_THRESHOLD
                )
            ),
        },

        "counterfactual_attribution": {
            "metric": "normalized_Jensen-Shannon_divergence",
            "interpretation": (
                "model dependence / counterfactual sensitivity; "
                "not physical causality"
            ),
            "mean_js_sar": mean_js_sar,
            "mean_js_optical": mean_js_opt,
            "sar_share": float(sar_share),
            "optical_share": float(optical_share),
            "dominance": dominance_label(
                sar_share,
                optical_share,
            ),
            "dominance_share_threshold": DOMINANCE_SHARE,
            "mean_fragility": float(fragility[coverage].mean()),
            "high_fragility_tile_threshold": HIGH_FRAGILITY_THRESHOLD,
            "high_fragility_tile_fraction": high_fragility_tile_fraction,
            "flip_rate_no_sar": float(
                sm["flip_no_sar"][coverage].mean()
            ),
            "flip_rate_no_optical": float(
                sm["flip_no_optical"][coverage].mean()
            ),
            "temporary_flood_js_sar": masked_mean(
                js_sar,
                temporary,
            ),
            "temporary_flood_js_optical": masked_mean(
                js_opt,
                temporary,
            ),
        },

        "spatial_geometry": {
            "temporary_flood_connected_components_8conn": (
                temp_geom["component_count"]
            ),
            "largest_temporary_component_pixels": (
                temp_geom["largest_component_pixels"]
            ),
            "largest_temporary_component_area_km2": (
                temp_geom["largest_component_area_km2"]
            ),
            "largest_temporary_component_fraction_of_temp": (
                temp_geom["largest_component_fraction_of_temp"]
            ),
            "largest_temporary_component_centroid": (
                temp_geom["largest_component_centroid"]
            ),
        },

        "visual_evidence": {
            "panel_file": f"panels/{scene_id}_evidence.png",
            "contains_ground_truth": False,
        },

        "grounding_policy": {
            "allowed": [
                "describe predicted class extent",
                "describe calibrated uncertainty",
                "describe P(clear) sensor reliability",
                "describe counterfactual SAR/optical model dependence",
                "describe deterministic spatial geometry",
            ],
            "prohibited": [
                "claim physical causality from counterfactual attribution",
                "claim prediction is ground truth",
                "invent flood cause, damage, population impact, or future evolution",
                "replace the segmentation detector with language-model judgement",
            ],
        },
    }

    # Compact VLM-facing structured facts.
    packet["fact_table"] = [
        {
            "fact_id": "temporary_flood_fraction",
            "value": packet["segmentation"]["temporary_flood"]["fraction_of_coverage"],
            "unit": "fraction_of_sensor_covered_area",
        },
        {
            "fact_id": "temporary_flood_area",
            "value": packet["segmentation"]["temporary_flood"]["area_km2"],
            "unit": "km2",
        },
        {
            "fact_id": "mean_predictive_entropy",
            "value": packet["uncertainty"]["mean_entropy"],
            "unit": "normalized_0_1",
        },
        {
            "fact_id": "review10_fraction",
            "value": packet["uncertainty"]["review10_fraction"],
            "unit": "fraction_of_sensor_covered_pixels",
        },
        {
            "fact_id": "mean_pclear",
            "value": packet["sensor_quality"]["mean_pclear"],
            "unit": "probability",
        },
        {
            "fact_id": "sar_attribution_share",
            "value": packet["counterfactual_attribution"]["sar_share"],
            "unit": "share",
        },
        {
            "fact_id": "optical_attribution_share",
            "value": packet["counterfactual_attribution"]["optical_share"],
            "unit": "share",
        },
        {
            "fact_id": "sensor_dominance",
            "value": packet["counterfactual_attribution"]["dominance"],
            "unit": "category",
        },
        {
            "fact_id": "high_fragility_tile_fraction",
            "value": packet["counterfactual_attribution"]["high_fragility_tile_fraction"],
            "unit": "fraction_of_four_tiles",
        },
    ]

    summary_rows.append({
        "scene_id": scene_id,
        "coverage_fraction": packet["coverage"]["coverage_fraction"],
        "coverage_area_km2": packet["coverage"]["coverage_area_km2"],
        "temporary_flood_fraction": packet["segmentation"]["temporary_flood"]["fraction_of_coverage"],
        "temporary_flood_area_km2": packet["segmentation"]["temporary_flood"]["area_km2"],
        "permanent_water_fraction": packet["segmentation"]["permanent_water"]["fraction_of_coverage"],
        "mean_entropy": packet["uncertainty"]["mean_entropy"],
        "review10_fraction": packet["uncertainty"]["review10_fraction"],
        "review20_fraction": packet["uncertainty"]["review20_fraction"],
        "mean_pclear": packet["sensor_quality"]["mean_pclear"],
        "low_pclear_fraction": packet["sensor_quality"]["low_pclear_fraction"],
        "mean_js_sar": mean_js_sar,
        "mean_js_optical": mean_js_opt,
        "sar_share": sar_share,
        "optical_share": optical_share,
        "dominance": packet["counterfactual_attribution"]["dominance"],
        "high_fragility_tile_fraction": high_fragility_tile_fraction,
        "flip_rate_no_sar": packet["counterfactual_attribution"]["flip_rate_no_sar"],
        "flip_rate_no_optical": packet["counterfactual_attribution"]["flip_rate_no_optical"],
        "temporary_components": temp_geom["component_count"],
        "largest_temp_component_area_km2": temp_geom["largest_component_area_km2"],
    })

    packets.append(packet)

summary_df = pd.DataFrame(summary_rows)

print("=" * 112)
print("P15-C12 — PACKETS BUILT")
print("=" * 112)
print("Packets:", len(packets))
print("Scenes :", len(summary_df))
print("Area available:", int(summary_df["coverage_area_km2"].notna().sum()), "/", len(summary_df))
print("Ground truth fields exported: 0")
print("P15-C12 COMPLETE ✅")


P15-C12 — PACKETS BUILT
Packets: 34
Scenes : 34
Area available: 34 / 34
Ground truth fields exported: 0
P15-C12 COMPLETE ✅


In [23]:
# ============================================================
# CELL: P15-C13 — DETERMINISTIC PACKET VERIFIER + SHA256
# ============================================================
def verify_packet(packet):
    errors = []

    seg = packet["segmentation"]
    cov = packet["coverage"]
    unc = packet["uncertainty"]
    att = packet["counterfactual_attribution"]

    # 1. Class fractions must partition covered pixels.
    class_fraction_sum = (
        seg["non_flood"]["fraction_of_coverage"]
        + seg["permanent_water"]["fraction_of_coverage"]
        + seg["temporary_flood"]["fraction_of_coverage"]
    )

    if abs(class_fraction_sum - 1.0) > 1e-6:
        errors.append(
            f"class fractions sum to {class_fraction_sum}, not 1"
        )

    # 2. Water total must equal permanent + temporary.
    expected_water_pixels = (
        seg["permanent_water"]["pixels"]
        + seg["temporary_flood"]["pixels"]
    )

    if seg["predicted_water_total"]["pixels"] != expected_water_pixels:
        errors.append("water pixel count mismatch")

    # 3. Counterfactual shares must sum to 1.
    if abs(att["sar_share"] + att["optical_share"] - 1.0) > 1e-6:
        errors.append("attribution shares do not sum to 1")

    # 4. Frozen review-10 is stricter than review-20.
    if unc["review10_fraction"] > unc["review20_fraction"] + 1e-12:
        errors.append("review10 fraction exceeds review20 fraction")

    # 5. Probability/fraction bounds.
    bounded_values = [
        cov["coverage_fraction"],
        seg["non_flood"]["fraction_of_coverage"],
        seg["permanent_water"]["fraction_of_coverage"],
        seg["temporary_flood"]["fraction_of_coverage"],
        unc["mean_entropy"],
        unc["review10_fraction"],
        unc["review20_fraction"],
        packet["sensor_quality"]["mean_pclear"],
        packet["sensor_quality"]["low_pclear_fraction"],
        att["sar_share"],
        att["optical_share"],
        att["mean_fragility"],
        att["high_fragility_tile_fraction"],
        att["flip_rate_no_sar"],
        att["flip_rate_no_optical"],
    ]

    for x in bounded_values:
        if x is not None and not (-1e-8 <= float(x) <= 1.0 + 1e-8):
            errors.append(f"bounded value outside [0,1]: {x}")

    # 6. Area conservation when geospatial area is available.
    if cov["coverage_area_km2"] is not None:
        class_area_sum = (
            seg["non_flood"]["area_km2"]
            + seg["permanent_water"]["area_km2"]
            + seg["temporary_flood"]["area_km2"]
        )

        if abs(
            class_area_sum
            - cov["coverage_area_km2"]
        ) > max(1e-8, 1e-6 * cov["coverage_area_km2"]):
            errors.append("class areas do not sum to coverage area")

    # 7. VLM-facing packet must not carry target/ground-truth keys.
    serialized = json.dumps(packet).lower()

    forbidden_tokens = [
        '"ground_truth"',
        '"target"',
        '"label_mask"',
        '"true_class"',
    ]

    for token in forbidden_tokens:
        if token in serialized:
            errors.append(f"forbidden packet token found: {token}")

    return errors

verified_packets = []

for packet in packets:
    errs = verify_packet(packet)

    if errs:
        raise AssertionError(
            f"Packet verification failed for {packet['scene_id']}: {errs}"
        )

    # Canonical provenance hash BEFORE inserting the hash field.
    canonical = json.dumps(
        packet,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")

    packet_sha = hashlib.sha256(canonical).hexdigest()

    packet["provenance"]["packet_sha256"] = packet_sha
    packet["provenance"]["verifier_status"] = "PASS"

    verified_packets.append(packet)

assert len(verified_packets) == 34

print("=" * 112)
print("P15-C13 — PACKET VERIFIER")
print("=" * 112)
print("Verified packets:", len(verified_packets))
print("Failures        : 0")
print("Ground truth    : excluded from all packets")
print("P15-C13 COMPLETE ✅")


P15-C13 — PACKET VERIFIER
Verified packets: 34
Failures        : 0
Ground truth    : excluded from all packets
P15-C13 COMPLETE ✅


In [24]:
# ============================================================
# CELL: P15-C14 — GROUNDING-SAFE EVIDENCE PANELS
#
# No ground truth is displayed.
# ============================================================
def stretch_channel(x, mask):
    vals = x[mask & np.isfinite(x)]

    if vals.size == 0:
        return np.zeros_like(x, dtype=np.float32)

    lo = np.quantile(vals, 0.02)
    hi = np.quantile(vals, 0.98)

    if hi <= lo:
        return np.zeros_like(x, dtype=np.float32)

    return np.clip(
        (x - lo) / (hi - lo),
        0.0,
        1.0,
    ).astype(np.float32)

def stretch_rgb(rgb, mask):
    out = np.zeros_like(rgb, dtype=np.float32)

    for c in range(3):
        out[..., c] = stretch_channel(
            rgb[..., c],
            mask,
        )

    out[~mask] = 0.0
    return out

for idx, packet in enumerate(verified_packets, start=1):
    sid = packet["scene_id"]
    sm = scene_maps[sid]
    cov = sm["coverage"].astype(bool)

    rgb = stretch_rgb(
        sm["rgb_reflectance"],
        cov,
    )

    vv = stretch_channel(
        sm["sar_vv_db"],
        cov,
    )

    frag = np.maximum(
        sm["js_sar"],
        sm["js_optical"],
    )

    fig, axes = plt.subplots(
        2,
        4,
        figsize=(14, 7),
    )

    axes[0, 0].imshow(rgb)
    axes[0, 0].set_title("Optical RGB")

    axes[0, 1].imshow(vv)
    axes[0, 1].set_title("SAR VV")

    axes[0, 2].imshow(sm["pclear"], vmin=0, vmax=1)
    axes[0, 2].set_title("P(clear)")

    axes[0, 3].imshow(sm["prediction"], vmin=0, vmax=2)
    axes[0, 3].set_title("04H-CAL prediction")

    axes[1, 0].imshow(sm["entropy"], vmin=0, vmax=1)
    axes[1, 0].set_title("Predictive entropy")

    axes[1, 1].imshow(sm["js_sar"], vmin=0, vmax=1)
    axes[1, 1].set_title("Counterfactual JS: SAR")

    axes[1, 2].imshow(sm["js_optical"], vmin=0, vmax=1)
    axes[1, 2].set_title("Counterfactual JS: optical")

    axes[1, 3].imshow(frag, vmin=0, vmax=1)
    axes[1, 3].set_title("Counterfactual fragility")

    for ax in axes.ravel():
        ax.axis("off")

    fig.suptitle(
        f"{sid} | deterministic FloodShift evidence",
        fontsize=12,
    )

    fig.tight_layout()

    out_path = PANEL_ROOT / f"{sid}_evidence.png"

    fig.savefig(
        out_path,
        dpi=120,
        bbox_inches="tight",
    )

    plt.close(fig)

    if idx == 1 or idx % 10 == 0 or idx == len(verified_packets):
        print(f"[{idx:02d}/{len(verified_packets)}] panels")

print("P15-C14 COMPLETE ✅")


[01/34] panels
[10/34] panels
[20/34] panels
[30/34] panels
[34/34] panels
P15-C14 COMPLETE ✅


In [25]:
# ============================================================
# CELL: P15-C15 — GEOENGINE SUMMARY
# ============================================================
print("=" * 112)
print("P15-C15 — GEOENGINE SUMMARY")
print("=" * 112)

print("Scenes:", len(summary_df))
print()
print("Sensor dominance:")
print(summary_df["dominance"].value_counts())
print()

print("Mean temporary-flood fraction:",
      float(summary_df["temporary_flood_fraction"].mean()))
print("Mean entropy:",
      float(summary_df["mean_entropy"].mean()))
print("Mean review-10 fraction:",
      float(summary_df["review10_fraction"].mean()))
print("Mean P(clear):",
      float(summary_df["mean_pclear"].mean()))
print("Mean optical attribution share:",
      float(summary_df["optical_share"].mean()))
print("Mean SAR attribution share:",
      float(summary_df["sar_share"].mean()))

display(
    summary_df[
        [
            "scene_id",
            "temporary_flood_fraction",
            "temporary_flood_area_km2",
            "mean_entropy",
            "review10_fraction",
            "mean_pclear",
            "sar_share",
            "optical_share",
            "dominance",
            "high_fragility_tile_fraction",
            "temporary_components",
        ]
    ]
    .sort_values(
        ["high_fragility_tile_fraction", "temporary_flood_fraction"],
        ascending=[False, False],
    )
    .head(12)
)

print("P15-C15 COMPLETE ✅")


P15-C15 — GEOENGINE SUMMARY
Scenes: 34

Sensor dominance:
dominance
Optical-dominant    30
Balanced             4
Name: count, dtype: int64

Mean temporary-flood fraction: 0.06875415839653144
Mean entropy: 0.09193312754744992
Mean review-10 fraction: 0.15120173499408465
Mean P(clear): 0.545442174134009
Mean optical attribution share: 0.790957657615024
Mean SAR attribution share: 0.20904234238497588


,scene_id,temporary_flood_fraction,temporary_flood_area_km2,mean_entropy,review10_fraction,mean_pclear,sar_share,optical_share,dominance,high_fragility_tile_fraction,temporary_components
24,Sri-Lanka_534068,0.063938,1.649183,0.188135,0.490215,0.579867,0.354575,0.645425,Balanced,1.00,30
29,Sri-Lanka_845821,0.035011,0.906537,0.207224,0.412270,0.728899,0.305296,0.694704,Optical-dominant,1.00,40
9,Somalia_886726,0.362350,9.405970,0.142592,0.278629,0.941366,0.133257,0.866743,Optical-dominant,0.50,26
0,Somalia_12849,0.206764,5.378528,0.186378,0.387241,0.497913,0.143555,0.856445,Optical-dominant,0.50,118
19,Sri-Lanka_14484,0.180805,4.682084,0.084113,0.130367,0.473045,0.057360,0.942640,Optical-dominant,0.50,42
27,Sri-Lanka_713926,0.033607,0.867312,0.083677,0.149670,0.995966,0.108713,0.891287,Optical-dominant,0.50,29
4,Somalia_322855,0.205353,5.339342,0.104203,0.188000,0.885961,0.104766,0.895234,Optical-dominant,0.25,25
32,Sri-Lanka_92824,0.199150,5.156238,0.137300,0.261536,0.517510,0.134638,0.865362,Optical-dominant,0.25,67
5,Somalia_371421,0.144188,3.741947,0.119508,0.214272,0.712999,0.047772,0.952228,Optical-dominant,0.25,62
30,Sri-Lanka_847275,0.130535,3.380325,0.165798,0.337616,0.225098,0.308000,0.692000,Optical-dominant,0.25,82


P15-C15 COMPLETE ✅


In [26]:
# ============================================================
# CELL: P15-C16 — EXPORT VERIFIED GROUNDED EVIDENCE
# ============================================================
PACKETS_JSONL = PACKET_ROOT / "FloodShift_07_scene_evidence_packets.jsonl"
SUMMARY_CSV = METRIC_ROOT / "FloodShift_07_scene_summary.csv"
SCHEMA_JSON = OUTPUT_ROOT / "FloodShift_07_fact_schema.json"
PROTOCOL_JSON = OUTPUT_ROOT / "FloodShift_07_protocol_record.json"

with open(PACKETS_JSONL, "w", encoding="utf-8") as f:
    for packet in verified_packets:
        f.write(
            json.dumps(
                packet,
                sort_keys=True,
                allow_nan=False,
            )
            + "\n"
        )

summary_df.to_csv(
    SUMMARY_CSV,
    index=False,
)

fact_schema = {
    "schema_version": PACKET_SCHEMA_VERSION,
    "scientific_role": (
        "deterministic sensor/model/geospatial evidence supplied to a later "
        "grounded language model"
    ),
    "packet_sections": [
        "provenance",
        "geospatial_reference",
        "coverage",
        "segmentation",
        "uncertainty",
        "sensor_quality",
        "counterfactual_attribution",
        "spatial_geometry",
        "visual_evidence",
        "grounding_policy",
        "fact_table",
    ],
    "never_include": [
        "ground truth",
        "test-set labels",
        "invented physical causes",
        "LLM-generated segmentation",
    ],
    "counterfactual_semantics": (
        "model dependence / sensitivity only; not physical causality"
    ),
}

with open(SCHEMA_JSON, "w", encoding="utf-8") as f:
    json.dump(
        fact_schema,
        f,
        indent=2,
    )

protocol07 = {
    "experiment": "FloodShift-07-Deterministic-GeoEngine",
    "source_segmenter": "FloodShift-04H-CAL",
    "source_04h_sha256": SHA_04H,
    "type_bias": TYPE_BIAS,
    "temperature": TEMPERATURE,
    "entropy_threshold_review_10": ENTROPY_THRESHOLD_REVIEW_10,
    "entropy_threshold_review_20": ENTROPY_THRESHOLD_REVIEW_20,
    "counterfactual_high_fragility_tile_threshold": HIGH_FRAGILITY_THRESHOLD,
    "counterfactual_dominance_share_threshold": DOMINANCE_SHARE,
    "source_uncertainty_stage": "FloodShift-05",
    "source_attribution_stage": "FloodShift-06",
    "05b_status": "NOT_PROMOTED_NOT_USED",
    "packet_count": len(verified_packets),
    "packet_schema_version": PACKET_SCHEMA_VERSION,
    "packet_verifier_failures": 0,
    "ground_truth_in_packet": False,
    "test_status": "LOCKED_NOT_EVALUATED",
    "next_stage": "grounded_instruction_dataset",
}

with open(PROTOCOL_JSON, "w", encoding="utf-8") as f:
    json.dump(
        protocol07,
        f,
        indent=2,
    )

archive_path = OUTPUT_ROOT / "FloodShift_07_ARCHIVE.zip"

with zipfile.ZipFile(
    archive_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:

    for p in [
        PACKETS_JSONL,
        SUMMARY_CSV,
        SCHEMA_JSON,
        PROTOCOL_JSON,
    ]:
        zf.write(
            p,
            arcname=p.relative_to(OUTPUT_ROOT),
        )

    for panel in sorted(PANEL_ROOT.glob("*.png")):
        zf.write(
            panel,
            arcname=panel.relative_to(OUTPUT_ROOT),
        )

archive_sha = sha256_file(archive_path)

print("=" * 112)
print("P15-C16 — EXPORT")
print("=" * 112)
print("Evidence packets :", PACKETS_JSONL)
print("Scene summary    :", SUMMARY_CSV)
print("Fact schema      :", SCHEMA_JSON)
print("Protocol record  :", PROTOCOL_JSON)
print("Evidence panels  :", len(list(PANEL_ROOT.glob('*.png'))))
print("Archive          :", archive_path)
print("Archive SHA256   :", archive_sha)
print("Ground truth     : NOT INCLUDED")
print("Frozen geo test  : LOCKED_NOT_EVALUATED")
print()
print("NEXT STAGE: grounded instruction dataset.")
print("P15-C16 COMPLETE ✅")


P15-C16 — EXPORT
Evidence packets : /kaggle/working/FloodShift_07_GeoEngine/packets/FloodShift_07_scene_evidence_packets.jsonl
Scene summary    : /kaggle/working/FloodShift_07_GeoEngine/metrics/FloodShift_07_scene_summary.csv
Fact schema      : /kaggle/working/FloodShift_07_GeoEngine/FloodShift_07_fact_schema.json
Protocol record  : /kaggle/working/FloodShift_07_GeoEngine/FloodShift_07_protocol_record.json
Evidence panels  : 34
Archive          : /kaggle/working/FloodShift_07_GeoEngine/FloodShift_07_ARCHIVE.zip
Archive SHA256   : 1a8a5c35f2afedeb92aa58d7c161bca22bbaee762d6533709a515f3798a8344b
Ground truth     : NOT INCLUDED
Frozen geo test  : LOCKED_NOT_EVALUATED

NEXT STAGE: grounded instruction dataset.
P15-C16 COMPLETE ✅


## After the run

Send me the executed notebook or these outputs:

- **P15-C10** — Notebook-06 integrity audit
- **P15-C12** — packet creation summary
- **P15-C13** — verifier result
- **P15-C15** — GeoEngine summary
- **P15-C16** — export paths + archive SHA

Download:

```text
FloodShift_07_ARCHIVE.zip
```

The most important next-stage artifacts inside it are:

```text
packets/FloodShift_07_scene_evidence_packets.jsonl
metrics/FloodShift_07_scene_summary.csv
FloodShift_07_fact_schema.json
FloodShift_07_protocol_record.json
panels/*.png
```

These form the deterministic grounding source for the next notebook.

The later language model is allowed to **describe these facts**, but it is not allowed to invent numerical values, replace the detector, or infer physical causality from the sensor-attribution scores.
